## SQL version of the Olist delivery analysis
The same analysis as the pandas notebook, rebuilt in SQL (SQLite inside Python) to verify every number independently.

In [ ]:
import pandas as pd
import sqlite3

con = sqlite3.connect(':memory:')
for name in ['orders', 'customers', 'order_items', 'sellers', 'order_reviews']:
    pd.read_csv(f'/content/olist_{name}_dataset.csv').to_sql(name, con, index=False)
pd.read_sql("SELECT COUNT(*) AS orders FROM orders", con)

,orders
0,99441


## 1. Overall late %
6.77% of delivered orders were late, matching the pandas analysis.

In [ ]:
pd.read_sql("""
WITH delivered AS (
    SELECT CASE WHEN date(order_delivered_customer_date) > date(order_estimated_delivery_date)
                THEN 1 ELSE 0 END AS late
    FROM orders
    WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
)
SELECT ROUND(100.0 * SUM(late) / COUNT(*), 2) AS late_pct
FROM delivered
""", con)

,late_pct
0,6.77


## 2. Late % by state
AL has the worst late rate (21.41%), followed by MA (17.43%), matching pandas.

In [ ]:
pd.read_sql("""
WITH delivered AS (
    SELECT order_id, customer_id,
           CASE WHEN date(order_delivered_customer_date) > date(order_estimated_delivery_date)
                THEN 1 ELSE 0 END AS late
    FROM orders
    WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
)
SELECT customers.customer_state,
       COUNT(*) AS orders,
       ROUND(100.0 * SUM(late) / COUNT(*), 2) AS late_pct
FROM delivered
JOIN customers ON delivered.customer_id = customers.customer_id
GROUP BY customers.customer_state
ORDER BY late_pct DESC
LIMIT 5
""", con)

,customer_state,orders,late_pct
0,AL,397,21.41
1,MA,717,17.43
2,SE,335,15.22
3,PI,476,13.87
4,CE,1279,13.76


## 3. Same-state vs cross-state sellers
Cross-state orders were late 8.05% of the time, against 4.50% for same-state orders, matching pandas. For orders with several sellers, the first item's seller is used.

In [ ]:
pd.read_sql("""
WITH delivered AS (
    SELECT order_id, customer_id,
           CASE WHEN date(order_delivered_customer_date) > date(order_estimated_delivery_date)
                THEN 1 ELSE 0 END AS late
    FROM orders
    WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
),
ranked AS (
    SELECT order_items.order_id, order_items.order_item_id, sellers.seller_state,
           ROW_NUMBER() OVER (PARTITION BY order_items.order_id ORDER BY order_item_id) AS rn
    FROM order_items
    JOIN sellers ON order_items.seller_id = sellers.seller_id
)
SELECT CASE WHEN customers.customer_state = ranked.seller_state THEN 'same state' ELSE 'cross state' END AS seller_location,
       COUNT(*) AS orders,
       ROUND(100.0 * SUM(late) / COUNT(*), 2) AS late_pct
FROM delivered
JOIN customers ON delivered.customer_id = customers.customer_id
JOIN ranked ON delivered.order_id = ranked.order_id
WHERE rn = 1
GROUP BY seller_location
""", con)

,seller_location,orders,late_pct
0,cross state,61769,8.05
1,same state,34701,4.50


## 4. Review scores: late vs on time
Late orders averaged 2.27 stars against 4.29 for on-time orders, matching pandas. Orders with several reviews were averaged, and orders with no review are kept with a LEFT JOIN.

In [ ]:
pd.read_sql("""
WITH delivered AS (
    SELECT order_id,
           CASE WHEN date(order_delivered_customer_date) > date(order_estimated_delivery_date)
                THEN 1 ELSE 0 END AS late
    FROM orders
    WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
),
average AS (
    SELECT order_id, AVG(review_score) AS review_score
    FROM order_reviews
    GROUP BY order_id
)
SELECT late,
       COUNT(*) AS orders,
       ROUND(AVG(average.review_score), 2) AS avg_score
FROM delivered
LEFT JOIN average ON delivered.order_id = average.order_id
GROUP BY late
""", con)

,late,orders,avg_score
0,0,89936,4.29
1,1,6534,2.27
